In [3]:
import pandas as pd
import os
from typing import List, Dict, Tuple
from difflib import SequenceMatcher
import re

# Install required packages if not already installed
try:
    import fitz  # PyMuPDF
except ImportError:
    print("Installing PyMuPDF...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyMuPDF"])
    import fitz

def extract_pdf_data(pdf_path: str) -> Tuple[str, Dict[int, str], fitz.Document]:
    """Extract text from PDF and return full text, page texts, and document."""
    try:
        doc = fitz.open(pdf_path)
        full_text, page_texts = "", {}
        for i in range(doc.page_count):
            page_text = doc[i].get_text()
            page_texts[i + 1] = page_text.lower()
            full_text += page_text
        return full_text.lower(), page_texts, doc
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return "", {}, None

def get_similarity(a: str, b: str) -> float:
    """Calculate similarity between two strings."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def find_fuzzy_matches(requirement: str, page_texts: Dict[int, str], threshold: float = 0.6, window_size: int = 100) -> List[Tuple[int, str, float, int]]:
    """Find fuzzy matches using sliding window approach."""
    req_lower = requirement.lower()
    req_words = req_lower.split()
    matches = []
    
    for page_num, page_text in page_texts.items():
        words = page_text.split()
        
        # Sliding window approach
        for i in range(len(words) - len(req_words) + 1):
            window = ' '.join(words[i:i + window_size])
            similarity = get_similarity(req_lower, window)
            
            if similarity >= threshold:
                matches.append((page_num, window, similarity, i))
    
    # Return top matches sorted by similarity
    return sorted(matches, key=lambda x: x[2], reverse=True)

def find_and_highlight_matches(requirement: str, page_texts: Dict[int, str], doc: fitz.Document, context: int = 200, fuzzy_threshold: float = 0.6) -> Dict:
    """Find exact and fuzzy matches, extract snippets, and highlight in PDF."""
    req_lower = requirement.lower()
    
    # Exact match results
    exact_pages, exact_snippets, exact_highlighted = [], {}, []
    
    # Fuzzy match results
    fuzzy_pages, fuzzy_snippets, fuzzy_highlighted = [], {}, []
    best_fuzzy_match = None
    
    # First, check for exact matches
    for page_num, page_text in page_texts.items():
        if req_lower in page_text:
            exact_pages.append(page_num)
            page_snippets = []
            page = doc[page_num - 1]
            original_text = page.get_text()
            
            start = 0
            while True:
                pos = page_text.find(req_lower, start)
                if pos == -1:
                    break
                    
                # Extract snippet with context
                snippet_start = max(0, pos - context)
                snippet_end = min(len(page_text), pos + len(req_lower) + context)
                snippet = ' '.join(page_text[snippet_start:snippet_end].split())
                
                if snippet_start > 0:
                    snippet = "..." + snippet
                if snippet_end < len(page_text):
                    snippet += "..."
                page_snippets.append(snippet)
                
                # Highlight in PDF (yellow for exact matches)
                snippet_text = original_text[snippet_start:snippet_end]
                text_instances = page.search_for(snippet_text.strip(), flags=fitz.TEXT_DEHYPHENATE)
                
                if not text_instances:
                    req_instances = page.search_for(requirement, flags=fitz.TEXT_DEHYPHENATE)
                    if not req_instances:
                        orig_pos = original_text.lower().find(req_lower)
                        if orig_pos != -1:
                            orig_req = original_text[orig_pos:orig_pos + len(requirement)]
                            req_instances = page.search_for(orig_req, flags=fitz.TEXT_DEHYPHENATE)
                    
                    for rect in req_instances:
                        expanded_rect = fitz.Rect(
                            max(0, rect.x0 - 100), max(0, rect.y0 - 30),
                            min(page.rect.width, rect.x1 + 100), min(page.rect.height, rect.y1 + 30)
                        )
                        highlight = page.add_highlight_annot(expanded_rect)
                        highlight.set_colors(stroke=[1, 1, 0])  # Yellow for exact
                        highlight.update()
                else:
                    for inst in text_instances:
                        highlight = page.add_highlight_annot(inst)
                        highlight.set_colors(stroke=[1, 1, 0])  # Yellow for exact
                        highlight.update()
                
                start = pos + 1
            
            if page_snippets:
                exact_snippets[page_num] = page_snippets
                if page_num not in exact_highlighted:
                    exact_highlighted.append(page_num)
    
    # If no exact matches, look for fuzzy matches
    if not exact_pages:
        fuzzy_matches = find_fuzzy_matches(requirement, page_texts, fuzzy_threshold)
        
        if fuzzy_matches:
            # Take the best fuzzy match
            best_match = fuzzy_matches[0]
            page_num, match_text, similarity, word_pos = best_match
            best_fuzzy_match = {
                'page': page_num,
                'text': match_text[:200] + "..." if len(match_text) > 200 else match_text,
                'similarity': similarity
            }
            
            fuzzy_pages.append(page_num)
            fuzzy_snippets[page_num] = [f"...{match_text[:200]}..." if len(match_text) > 200 else match_text]
            
            # Highlight fuzzy match in PDF (orange)
            page = doc[page_num - 1]
            original_text = page.get_text()
            
            # Try to find the fuzzy match text in the original PDF
            text_instances = page.search_for(match_text[:50], flags=fitz.TEXT_DEHYPHENATE)  # Search first 50 chars
            
            if text_instances:
                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.set_colors(stroke=[1, 0.5, 0])  # Orange for fuzzy
                    highlight.update()
                fuzzy_highlighted.append(page_num)
            else:
                # Fallback: highlight area based on word position
                words = original_text.split()
                if word_pos < len(words):
                    search_text = ' '.join(words[word_pos:word_pos + 10])  # 10 words
                    instances = page.search_for(search_text, flags=fitz.TEXT_DEHYPHENATE)
                    for inst in instances:
                        highlight = page.add_highlight_annot(inst)
                        highlight.set_colors(stroke=[1, 0.5, 0])  # Orange for fuzzy
                        highlight.update()
                    if instances:
                        fuzzy_highlighted.append(page_num)
    
    return {
        'exact_match': bool(exact_pages),
        'exact_pages': exact_pages,
        'exact_snippets': exact_snippets,
        'exact_highlighted_pages': exact_highlighted,
        'fuzzy_match': bool(fuzzy_pages),
        'fuzzy_pages': fuzzy_pages,
        'fuzzy_snippets': fuzzy_snippets,
        'fuzzy_highlighted_pages': fuzzy_highlighted,
        'best_fuzzy_match': best_fuzzy_match
    }

def check_document_requirements(document_id: int, csv_path: str, documents_folder: str, output_folder: str, fuzzy_threshold: float = 0.6):
    """Main function to check requirements and generate outputs."""
    # Load data
    try:
        df = pd.read_csv(csv_path)
        doc_requirements = df[df['document_id'] == document_id].copy()
        if doc_requirements.empty:
            print(f"No requirements found for document_id {document_id}")
            return
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return
    
    # Filter for unique requirements only
    original_count = len(doc_requirements)
    doc_requirements['requirement_clean'] = doc_requirements['requirement'].astype(str).str.strip().str.lower()
    unique_requirements = doc_requirements.drop_duplicates(subset=['requirement_clean'], keep='first')
    unique_count = len(unique_requirements)
    
    print(f"Original requirements: {original_count}")
    print(f"Unique requirements: {unique_count}")
    print(f"Duplicates removed: {original_count - unique_count}")
    
    # Load PDF
    pdf_path = os.path.join(documents_folder, f"{document_id}.pdf")
    if not os.path.exists(pdf_path):
        print(f"PDF not found: {pdf_path}")
        return
    
    pdf_text, page_texts, doc = extract_pdf_data(pdf_path)
    if not doc:
        return
    
    print(f"Processing {unique_count} unique requirements from {len(page_texts)} pages...")
    
    # Process unique requirements
    exact_results = []
    fuzzy_results = []
    any_highlights = False
    
    for idx, row in unique_requirements.iterrows():
        requirement = str(row['requirement'])
        if pd.isna(requirement) or not requirement.strip():
            continue
        
        result = find_and_highlight_matches(requirement, page_texts, doc, fuzzy_threshold=fuzzy_threshold)
        
        if result['exact_highlighted_pages'] or result['fuzzy_highlighted_pages']:
            any_highlights = True
        
        # Common data for both exact and fuzzy results
        common_data = {
            'row_index': idx, 'document_id': document_id,
            'model_name': row.get('model_name', ''),
            'constraint_type': row.get('constraint_type', ''),
            'scope': row.get('scope', ''),
            'numerical_value': row.get('numerical_value', ''),
            'unit': row.get('unit', ''),
            'requirement': requirement
        }
        
        # Process exact matches
        if result['exact_match']:
            exact_snippets_text = ""
            for page_num in sorted(result['exact_snippets'].keys()):
                exact_snippets_text += f"Page {page_num}: "
                exact_snippets_text += " | ".join(f'"{s}"' for s in result['exact_snippets'][page_num])
                exact_snippets_text += "\n"
            
            exact_results.append({
                **common_data,
                'match_type': 'exact',
                'pages_found': ', '.join(map(str, result['exact_pages'])),
                'highlighted_pages': ', '.join(map(str, result['exact_highlighted_pages'])),
                'matching_text_snippets': exact_snippets_text.strip(),
                'similarity_score': 1.0
            })
        
        # Process fuzzy matches (only if no exact match)
        elif result['fuzzy_match']:
            fuzzy_snippets_text = ""
            for page_num in sorted(result['fuzzy_snippets'].keys()):
                fuzzy_snippets_text += f"Page {page_num}: "
                fuzzy_snippets_text += " | ".join(f'"{s}"' for s in result['fuzzy_snippets'][page_num])
                fuzzy_snippets_text += "\n"
            
            fuzzy_results.append({
                **common_data,
                'match_type': 'fuzzy',
                'pages_found': ', '.join(map(str, result['fuzzy_pages'])),
                'highlighted_pages': ', '.join(map(str, result['fuzzy_highlighted_pages'])),
                'matching_text_snippets': fuzzy_snippets_text.strip(),
                'similarity_score': result['best_fuzzy_match']['similarity'] if result['best_fuzzy_match'] else 0.0
            })
        else:
            # No match found
            exact_results.append({
                **common_data,
                'match_type': 'none',
                'pages_found': '',
                'highlighted_pages': '',
                'matching_text_snippets': '',
                'similarity_score': 0.0
            })
    
    # Create DataFrames
    exact_df = pd.DataFrame(exact_results)
    fuzzy_df = pd.DataFrame(fuzzy_results)
    combined_df = pd.concat([exact_df, fuzzy_df], ignore_index=True) if not fuzzy_df.empty else exact_df
    
    # Save outputs
    os.makedirs(output_folder, exist_ok=True)
    
    if any_highlights:
        highlighted_pdf = os.path.join(output_folder, f"document_{document_id}_highlighted.pdf")
        doc.save(highlighted_pdf)
        print(f"Highlighted PDF: {highlighted_pdf}")
    
    doc.close()
    
    # Statistics
    total = len(exact_df)
    exact_matches = exact_df[exact_df['match_type'] == 'exact'].shape[0]
    fuzzy_matches = len(fuzzy_df)
    exact_highlighted = exact_df[exact_df['highlighted_pages'] != ''].shape[0]
    fuzzy_highlighted = fuzzy_df[fuzzy_df['highlighted_pages'] != ''].shape[0] if not fuzzy_df.empty else 0
    
    print(f"\n=== SUMMARY ===")
    print(f"Unique requirements processed: {total}")
    print(f"Exact matches: {exact_matches} (highlighted: {exact_highlighted})")
    print(f"Fuzzy matches: {fuzzy_matches} (highlighted: {fuzzy_highlighted})")
    print(f"Not found: {total - exact_matches - fuzzy_matches}")
    print(f"Overall success: {(exact_matches + fuzzy_matches)/total*100:.1f}%")
    
    # Save separate results
    exact_df.to_csv(os.path.join(output_folder, f"document_{document_id}_exact_results_unique.csv"), index=False)
    if not fuzzy_df.empty:
        fuzzy_df.to_csv(os.path.join(output_folder, f"document_{document_id}_fuzzy_results_unique.csv"), index=False)
    combined_df.to_csv(os.path.join(output_folder, f"document_{document_id}_combined_results_unique.csv"), index=False)
    
    # Save detailed summary
    with open(os.path.join(output_folder, f"document_{document_id}_detailed_summary_unique.txt"), 'w') as f:
        f.write(f"DETAILED SUMMARY FOR DOCUMENT {document_id} (UNIQUE REQUIREMENTS)\n{'='*70}\n")
        f.write(f"Original requirements: {original_count}\n")
        f.write(f"Unique requirements processed: {total}\n")
        f.write(f"Duplicates removed: {original_count - unique_count}\n")
        f.write(f"Exact matches: {exact_matches} (highlighted: {exact_highlighted})\n")
        f.write(f"Fuzzy matches: {fuzzy_matches} (highlighted: {fuzzy_highlighted})\n")
        f.write(f"Not found: {total - exact_matches - fuzzy_matches}\n")
        f.write(f"Overall success: {(exact_matches + fuzzy_matches)/total*100:.1f}%\n\n")
        
        f.write("EXACT MATCHES:\n" + "-"*30 + "\n")
        for _, row in exact_df[exact_df['match_type'] == 'exact'].iterrows():
            f.write(f"✓ {row['requirement'][:80]}...\n")
            if row['matching_text_snippets']:
                f.write(f"   {row['matching_text_snippets'][:100]}...\n")
            f.write("\n")
        
        if not fuzzy_df.empty:
            f.write("\nFUZZY MATCHES:\n" + "-"*30 + "\n")
            for _, row in fuzzy_df.iterrows():
                f.write(f"≈ {row['requirement'][:80]}... (similarity: {row['similarity_score']:.2f})\n")
                if row['matching_text_snippets']:
                    f.write(f"   {row['matching_text_snippets'][:100]}...\n")
                f.write("\n")
        
        not_found_df = exact_df[exact_df['match_type'] == 'none']
        if not not_found_df.empty:
            f.write("\nNOT FOUND:\n" + "-"*30 + "\n")
            for _, row in not_found_df.iterrows():
                f.write(f"✗ {row['requirement'][:80]}...\n")
    
    print(f"Results saved to: {output_folder}")
    print("Exact matches highlighted in YELLOW")
    print("Fuzzy matches highlighted in ORANGE")
    print(f"Note: Processing focused on {unique_count} unique requirements out of {original_count} total")

In [8]:
# Configuration and execution
DOCUMENT_ID = 10
CSV_PATH = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/combined_all_filtered_constraints.csv"
DOCUMENTS_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents"
OUTPUT_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result"
FUZZY_THRESHOLD = 0.6  # Minimum similarity score for fuzzy matches (0.0-1.0)

if __name__ == "__main__":
    check_document_requirements(DOCUMENT_ID, CSV_PATH, DOCUMENTS_FOLDER, OUTPUT_FOLDER, FUZZY_THRESHOLD)

Original requirements: 348
Unique requirements: 307
Duplicates removed: 41
Processing 307 unique requirements from 376 pages...
Processing 307 unique requirements from 376 pages...
Highlighted PDF: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result/document_10_highlighted.pdf

=== SUMMARY ===
Unique requirements processed: 191
Exact matches: 4 (highlighted: 4)
Fuzzy matches: 116 (highlighted: 115)
Not found: 71
Overall success: 62.8%
Results saved to: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result
Exact matches highlighted in YELLOW
Fuzzy matches highlighted in ORANGE
Note: Processing focused on 307 unique requirements out of 348 total
Highlighted PDF: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result/document_10_highlighted.pdf

=== SUMMARY ===
Unique requirements proces